# 第 9 章: 特徴量エンジニアリングの探索と可視化

標準化の前後の分布、特徴量と価格の関係、価格の外れ値、天気ごとの利用者数を確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter09.COLUMNS
import chapter09.Standardizer
import chapter09.iqrOutliers
import chapter09.joinWeather
import chapter09.loadBike
import chapter09.loadWeather
import chapter09.meanCountByWeather
import chapter09.prepareBoston
import chapter09.quantile
import java.io.File

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val dataDirectory = dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }
val split = prepareBoston(File(dataDirectory, "Boston.csv"), testSize = 0.3, seed = 0)

## 標準化の前後で分布を比べる

In [ ]:
val train = split.xTrain.select("RM", "LSTAT", "PTRATIO")
val standardized = Standardizer.fit(train).transform(train)
train.rowsCount() to train.columnsCount()

In [ ]:
dataFrameOf(
    "標準化前" to train["RM"].values().map { (it as Number).toDouble() },
    "標準化後" to standardized["RM"].values().map { it as Double },
).plot {
    points {
        x("標準化前")
        y("標準化後")
    }
    layout.title = "RM の標準化の前後"
}

In [ ]:
dataFrameOf(
    "列" to COLUMNS,
    "標準化前の平均" to COLUMNS.map { train[it].values().map { v -> (v as Number).toDouble() }.average() },
    "標準化後の標準偏差" to COLUMNS.map { standardized[it].cast<Double>().std(ddof = 0) },
)

## 特徴量と価格の関係を見る

In [ ]:
val priced = split.xTrain.add("PRICE") { split.tTrain[index()] }
priced.plot {
    points {
        x("RM")
        y("PRICE")
    }
    layout.title = "訓練データの RM と PRICE"
}

In [ ]:
priced.plot {
    points {
        x("LSTAT")
        y("PRICE")
    }
    layout.title = "訓練データの LSTAT と PRICE"
}

In [ ]:
fun pearson(xs: List<Double>, ys: List<Double>): Double {
    val mx = xs.average()
    val my = ys.average()
    val cov = xs.zip(ys).sumOf { (x, y) -> (x - mx) * (y - my) }
    return cov / Math.sqrt(xs.sumOf { (it - mx) * (it - mx) } * ys.sumOf { (it - my) * (it - my) })
}

val features = split.xTrain.columnNames()
dataFrameOf(
    "列" to features,
    "PRICE との相関係数" to features.map { name -> pearson(split.xTrain[name].values().map { (it as Number).toDouble() }, split.tTrain) },
).sortByDesc { "PRICE との相関係数"<Double>().map { Math.abs(it) } }

## 外れ値を見る

In [ ]:
val q1 = quantile(split.tTrain, 0.25)
val q3 = quantile(split.tTrain, 0.75)
val outliers = split.tTrain.filterIndexed { i, _ -> iqrOutliers(split.tTrain)[i] }
dataFrameOf(
    "外れ値の件数" to listOf(outliers.size),
    "高い側" to listOf(outliers.count { it > q3 }),
    "低い側" to listOf(outliers.count { it < q1 }),
)

In [ ]:
val flags = iqrOutliers(split.tTrain)
dataFrameOf(
    "順位" to split.tTrain.indices.sortedBy { split.tTrain[it] }.indices.toList(),
    "PRICE" to split.tTrain.sorted(),
    "外れ値" to split.tTrain.indices.sortedBy { split.tTrain[it] }.map { if (flags[it]) "外れ値" else "それ以外" },
).plot {
    points {
        x("順位")
        y("PRICE")
        color("外れ値")
    }
    layout.title = "訓練データの PRICE（小さい順）"
}

## 天気ごとの利用者数を見る

In [ ]:
val joined = joinWeather(loadBike(File(dataDirectory, "bike.tsv")), loadWeather(File(dataDirectory, "weather.csv")))
val means = meanCountByWeather(joined)
dataFrameOf("天気" to means.keys.toList(), "平均利用者数" to means.values.toList()).plot {
    bars {
        x("天気")
        y("平均利用者数")
    }
    layout.title = "天気ごとの平均利用者数"
}